# Dummy LSTM Language Model Training

This notebook trains a **simple LSTM as a baseline language model** on:
- **DailyDialog**: Conversational dialogs with emotions and acts  
- **Surgical Robotics**: Robot control instruction-response pairs

This is a **dummy/baseline model** to compare against GPT-2.

## Model Characteristics:
- Simple LSTM architecture (~2-5M parameters vs GPT-2's 124M)
- Basic SGD optimizer (no fancy optimizations)
- No weight decay, no LR scheduling, no early stopping
- Perfect for establishing a baseline!

## Configuration

In [ ]:
# Hardcoded configuration (from Dummy.yaml)
config = {
    # Data Parameters
    'max_length': 256,  # Reduced from 512 to save memory
    'batch_size': 2,    # Reduced from 8 to prevent OOM
    
    # Training Parameters
    'learning_rate': 0.01,  # Higher LR for SGD
    'num_epochs': 10,
    
    # Model Architecture
    'embed_dim': 256,
    'hidden_dim': 512,
    'num_layers': 2,
    'dropout': 0.3,
    
    # Evaluation Parameters
    'eval_sample_size': 100,
    'generation_max_length': 150,
    'generation_temperature': 0.7,
    'generation_top_p': 0.9,
    
    # Robot Control Split
    'robot_train_split': 0.7,
}

print("Configuration loaded:")
print(f"  Learning Rate: {config['learning_rate']}")
print(f"  Epochs: {config['num_epochs']}")
print(f"  Batch Size: {config['batch_size']}")
print(f"  Max Length: {config['max_length']}")

## Imports and Setup

In [ ]:
import os
import sys
import json
import random
import numpy as np
import torch
import torch.nn as nn
from torch.optim import SGD
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

# Add parent directory to path to import custom modules
sys.path.append(os.path.join(os.path.dirname(os.path.abspath('__file__')), '..', 'src'))

# Import the DummyLSTM model
from models.dummy_LSTM import DummyLSTM

# Set random seeds for reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

print("Imports complete!")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"MPS available: {torch.backends.mps.is_available() if hasattr(torch.backends, 'mps') else False}")

## Device Setup and Tokenizer

In [ ]:
# Device setup (MPS for Mac, CUDA for GPU, CPU otherwise)
if torch.cuda.is_available():
    device = torch.device('cuda')
    print("Using CUDA")
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = torch.device('mps')
    print("Using MPS (Apple Silicon)")
else:
    device = torch.device('cpu')
    print("Using CPU")

# Load tokenizer
print("\nLoading GPT-2 tokenizer...")
tokenizer = AutoTokenizer.from_pretrained('openai-community/gpt2')
tokenizer.pad_token = tokenizer.eos_token
print(f"Vocabulary size: {tokenizer.vocab_size}")
print(f"Pad token ID: {tokenizer.pad_token_id}")

## Data Loading Functions

In [ ]:
def load_daily_dialog(dataset_dir, split='train'):
    """
    Load DailyDialog dataset from text files.
    Returns list of tuples: (context, response)
    """
    dialogues_path = os.path.join(dataset_dir, split, f'dialogues_{split}.txt')
    
    if not os.path.exists(dialogues_path):
        raise FileNotFoundError(f"File not found: {dialogues_path}")
    
    with open(dialogues_path, 'r', encoding='utf-8') as f:
        dialogues = f.readlines()
    
    dialog_pairs = []
    for dialogue in dialogues:
        utterances = dialogue.strip().split('__eou__')
        utterances = [u.strip() for u in utterances if u.strip()]
        
        for i in range(len(utterances) - 1):
            context = utterances[i]
            response = utterances[i + 1]
            if context and response:
                dialog_pairs.append((context, response))
    
    return dialog_pairs

def load_robot_control(json_path):
    """
    Load robot control data from JSON file.
    Returns list of tuples: (phase, control_command)
    """
    with open(json_path, 'r') as f:
        data = json.load(f)
    
    robot_pairs = []
    for entry in data:
        phase = entry.get('phase', '')
        control = entry.get('control_command', '')
        if phase and control:
            robot_pairs.append((phase, control))
    
    return robot_pairs

print("Data loading functions defined!")

## Load DailyDialog Dataset

In [ ]:
# Set paths relative to notebook location
notebook_dir = os.path.abspath('')
src_dir = os.path.join(notebook_dir, '..', 'src')
daily_dialog_dir = os.path.join(src_dir, 'dataset', 'DailyDialog')

print("Loading DailyDialog dataset...")
print(f"Dataset directory: {daily_dialog_dir}")

# Load training and validation data
train_dialog = load_daily_dialog(daily_dialog_dir, split='train')
val_dialog = load_daily_dialog(daily_dialog_dir, split='validation')

print(f"\nDailyDialog loaded:")
print(f"  Training pairs: {len(train_dialog)}")
print(f"  Validation pairs: {len(val_dialog)}")
print(f"\nExample training pair:")
print(f"  Context: {train_dialog[0][0]}")
print(f"  Response: {train_dialog[0][1]}")

## Load Robot Control Dataset

In [ ]:
robot_control_path = os.path.join(src_dir, 'dataset', 'Surgical_Robotics', 'robot_control.json')

print("Loading Robot Control dataset...")
print(f"Dataset file: {robot_control_path}")

robot_data = load_robot_control(robot_control_path)

print(f"\nRobot Control loaded:")
print(f"  Total pairs: {len(robot_data)}")
print(f"\nExample robot control pair:")
print(f"  Phase: {robot_data[0][0]}")
print(f"  Control: {robot_data[0][1]}")

# Split robot data into train/val
robot_train_split = config.get('robot_train_split', 0.7)
split_idx = int(len(robot_data) * robot_train_split)
train_robot = robot_data[:split_idx]
val_robot = robot_data[split_idx:]

print(f"\nRobot Control split:")
print(f"  Training pairs: {len(train_robot)}")
print(f"  Validation pairs: {len(val_robot)}")

## Combine and Balance Datasets

In [ ]:
# Balance datasets by upsampling robot control data
def balance_datasets(dialog_data, robot_data):
    """Upsample robot data to balance with dialog data"""
    if len(robot_data) < len(dialog_data):
        # Calculate how many times to repeat robot data
        repeat_factor = len(dialog_data) // len(robot_data)
        remainder = len(dialog_data) % len(robot_data)
        
        balanced_robot = robot_data * repeat_factor + robot_data[:remainder]
        return dialog_data, balanced_robot
    else:
        return dialog_data, robot_data

# Balance training data
train_dialog_balanced, train_robot_balanced = balance_datasets(train_dialog, train_robot)

# Combine datasets
train_combined = train_dialog_balanced + train_robot_balanced
val_combined = val_dialog + val_robot

# Shuffle training data
random.shuffle(train_combined)

print("Dataset balancing complete!")
print(f"\nTraining data:")
print(f"  Dialog pairs: {len(train_dialog_balanced)}")
print(f"  Robot pairs: {len(train_robot_balanced)}")
print(f"  Total training pairs: {len(train_combined)}")
print(f"\nValidation data:")
print(f"  Dialog pairs: {len(val_dialog)}")
print(f"  Robot pairs: {len(val_robot)}")
print(f"  Total validation pairs: {len(val_combined)}")

## Create PyTorch Dataset Class

In [ ]:
class DialogDataset(Dataset):
    """PyTorch Dataset for dialog pairs"""
    
    def __init__(self, pairs, tokenizer, max_length=512):
        self.pairs = pairs
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.pairs)
    
    def __getitem__(self, idx):
        context, response = self.pairs[idx]
        
        # Format as "Context: ... Response: ..."
        text = f"Context: {context} Response: {response}"
        
        # Tokenize
        encoding = self.tokenizer(
            text,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        input_ids = encoding['input_ids'].squeeze()
        attention_mask = encoding['attention_mask'].squeeze()
        
        return {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'labels': input_ids.clone()  # For language modeling
        }

# Create datasets
max_length = config.get('max_length', 512)
train_dataset = DialogDataset(train_combined, tokenizer, max_length)
val_dataset = DialogDataset(val_combined, tokenizer, max_length)

print("PyTorch datasets created!")
print(f"  Training samples: {len(train_dataset)}")
print(f"  Validation samples: {len(val_dataset)}")

## Create DataLoaders

In [ ]:
batch_size = config.get('batch_size', 8)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0  # Set to 0 for notebook compatibility
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0
)

print("DataLoaders created!")
print(f"  Batch size: {batch_size}")
print(f"  Training batches: {len(train_loader)}")
print(f"  Validation batches: {len(val_loader)}")

## Initialize Model and Optimizer

In [ ]:
# Initialize DummyLSTM model
print("Initializing DummyLSTM model...")
model = DummyLSTM(
    vocab_size=tokenizer.vocab_size,
    embed_dim=config.get('embed_dim', 256),
    hidden_dim=config.get('hidden_dim', 512),
    num_layers=config.get('num_layers', 2),
    dropout=config.get('dropout', 0.3),
    pad_token_id=tokenizer.pad_token_id
).to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\nModel initialized:")
print(f"  Total parameters: {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")
print(f"  Device: {device}")

# Initialize optimizer (SGD - most basic)
learning_rate = config.get('learning_rate', 0.01)
optimizer = SGD(model.parameters(), lr=learning_rate)

print(f"\nOptimizer: SGD with lr={learning_rate}")

## Training Loop

In [ ]:
# Training history
train_losses = []
val_losses = []
num_epochs = config.get('num_epochs', 10)

# Clear GPU cache and restart kernel if needed
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"GPU Memory before training: {torch.cuda.memory_allocated()/1e9:.2f} GB")

print(f"Starting training for {num_epochs} epochs...")
print("="*60)

for epoch in range(num_epochs):
    # Training phase
    model.train()
    epoch_train_loss = 0.0
    
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Train]")
    for i, batch in enumerate(progress_bar):
        try:
            # Zero gradients first
            optimizer.zero_grad()
            
            # Move to device
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            # Forward pass
            outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            
            # Backward pass
            loss.backward()
            optimizer.step()
            
            # Track loss
            loss_val = loss.item()
            epoch_train_loss += loss_val
            
            # Aggressive cleanup
            del input_ids, attention_mask, labels, outputs, loss
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            
            progress_bar.set_postfix({'loss': f'{loss_val:.4f}'})
            
        except RuntimeError as e:
            if "out of memory" in str(e):
                print(f"\nOOM at batch {i}. Clearing cache and skipping...")
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
                continue
            else:
                raise e
    
    avg_train_loss = epoch_train_loss / len(train_loader)
    train_losses.append(avg_train_loss)
    
    # Validation phase
    model.eval()
    epoch_val_loss = 0.0
    
    with torch.no_grad():
        progress_bar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Val]")
        for batch in progress_bar:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            
            loss_val = loss.item()
            epoch_val_loss += loss_val
            
            del input_ids, attention_mask, labels, outputs, loss
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            
            progress_bar.set_postfix({'loss': f'{loss_val:.4f}'})
    
    avg_val_loss = epoch_val_loss / len(val_loader)
    val_losses.append(avg_val_loss)
    
    print(f"\nEpoch {epoch+1}/{num_epochs}:")
    print(f"  Train Loss: {avg_train_loss:.4f}")
    print(f"  Val Loss: {avg_val_loss:.4f}")
    print("="*60)
    
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\nTraining complete!")

## Plot Training Curves

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(range(1, num_epochs + 1), train_losses, label='Training Loss', marker='o')
plt.plot(range(1, num_epochs + 1), val_losses, label='Validation Loss', marker='s')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('DummyLSTM Training and Validation Loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nFinal Training Loss: {train_losses[-1]:.4f}")
print(f"Final Validation Loss: {val_losses[-1]:.4f}")

## Save Model

In [ ]:
# Create results directory
results_dir = os.path.join(src_dir, 'results', 'dummy_results')
os.makedirs(results_dir, exist_ok=True)

# Save final model
final_model_path = os.path.join(results_dir, 'dummy_lstm_final')
model.save_pretrained(final_model_path)
tokenizer.save_pretrained(final_model_path)

print(f"Model saved to: {final_model_path}")

# Save training history
history_path = os.path.join(results_dir, 'training_history.json')
with open(history_path, 'w') as f:
    json.dump({
        'train_losses': train_losses,
        'val_losses': val_losses,
        'num_epochs': num_epochs,
        'config': config
    }, f, indent=2)

print(f"Training history saved to: {history_path}")

## Generate Sample Outputs

In [ ]:
# Test generation with dialog example
dialog_prompt = "Context: How are you doing today?"
print(f"Dialog Prompt: {dialog_prompt}")
print("-" * 60)

input_ids = tokenizer.encode(dialog_prompt, return_tensors='pt').to(device)
generated_ids = model.generate(
    input_ids,
    max_length=config.get('generation_max_length', 150),
    temperature=config.get('generation_temperature', 0.7),
    top_p=config.get('generation_top_p', 0.9),
    pad_token_id=tokenizer.pad_token_id
)
generated_text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
print(f"Generated: {generated_text}\n")

# Test generation with robotics example  
robotics_prompt = "Context: Preparation Phase"
print(f"Robotics Prompt: {robotics_prompt}")
print("-" * 60)

input_ids = tokenizer.encode(robotics_prompt, return_tensors='pt').to(device)
generated_ids = model.generate(
    input_ids,
    max_length=config.get('generation_max_length', 150),
    temperature=config.get('generation_temperature', 0.7),
    top_p=config.get('generation_top_p', 0.9),
    pad_token_id=tokenizer.pad_token_id
)
generated_text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
print(f"Generated: {generated_text}")

## Complete! 🎉

The DummyLSTM baseline model has been trained successfully. You can now compare its performance with the GPT-2 Core model to understand the benefits of using a pre-trained transformer architecture.